In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator

# visualization
import seaborn as sns
import matplotlib.pyplot as plt

from pyspark.ml.feature import VectorAssembler

In [ ]:
LABEL_COLUMN='price'
SEED=42

spark = SparkSession.builder \
    .appName("Airbnb") \
    .config("spark.driver.memory", "24g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

raw_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("multiLine", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .option("mode", "PERMISSIVE")
    .parquet("../data/processed/airbnb_rio/total_data.parquet")
)

raw_df.createOrReplaceTempView('raw_listing')

In [3]:
# as the dataset is has a lot of columns, we did an analysis and selected the most 
# good appearing to the prediction. 

selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        calculated_host_listings_count AS host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification, 
        snapshot_month AS month,
        security_deposit,
        cleaning_fee
    FROM raw_listing
""")

selected_df.createOrReplaceTempView("selected_df")

## Initial Parsing

In [4]:
selected_df.show()

+-------------------+-------------------+-----------------+-------------------+-------------+------------+---------+--------+----+--------------------+------+-----------------------------+--------------------------------+-----+----------------+------------+
|           latitude|          longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|           amenities| price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|
+-------------------+-------------------+-----------------+-------------------+-------------+------------+---------+--------+----+--------------------+------+-----------------------------+--------------------------------+-----+----------------+------------+
|          -22.99195|          -43.43592|              0.0|                2.0|        House|         2.0|      2.0|     1.0| 3.0|{TV,Wifi,"Air con...| 250.0|                          0.0|                             0.0|  8.0

Shape of the dataset:

In [5]:
# change this to pure SQL
print(f'Rows: {selected_df.count()}, Columns: {len(selected_df.columns)}')

Rows: 784121, Columns: 16


Checking for Null's:

In [6]:
selected_df.createOrReplaceTempView("selected_df")
null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()

+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+
|       0|        0|              385|                  0|            0|           0|     1493|     775|2334|        0|    0|                            0|                               0|    0|          361062|      269335|
+--------+---------+-----------------+-------------------+-------------+------------+---------+-----

As we can see there's a lot of missing values in the `security_deposit` and `cleaning_fee` columns.

Dropping them from the database seems to be a good decision.

In [7]:
rows_before_drop = selected_df.count()
selected_df = selected_df.drop('security_deposit', 'cleaning_fee')

In [8]:
selected_df = selected_df.dropna()
print(f"({selected_df.count()}, {len(selected_df.columns)}) - {rows_before_drop - selected_df.count()} rows dropped")

(780036, 14) - 4085 rows dropped


Checking if there is any missing left:

In [9]:
selected_df.createOrReplaceTempView("selected_df")
null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()


+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|       0|        0|                0|                  0|            0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+



In [10]:
selected_df.createOrReplaceTempView('selected_df')

### Checking data types

In [11]:
selected_df.printSchema()

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- host_is_superhost: double (nullable = true)
 |-- host_listings_count: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- accommodates: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- beds: double (nullable = true)
 |-- amenities: string (nullable = true)
 |-- price: double (nullable = true)
 |-- require_guest_profile_picture: double (nullable = true)
 |-- require_guest_phone_verification: double (nullable = true)
 |-- month: double (nullable = true)



TODO: A brief explanation of what need to be changed

### Checking data types

In [12]:
selected_df = spark.sql("""
    SELECT
        TRY_CAST(latitude AS DOUBLE) AS latitude,
        TRY_CAST(longitude AS DOUBLE) AS longitude,
        TRY_CAST(host_is_superhost AS BOOLEAN) AS host_is_superhost,
        TRY_CAST(TRY_CAST(host_listings_count AS DOUBLE) AS INT) AS host_listings_count,
        property_type,
        TRY_CAST(TRY_CAST(accommodates AS DOUBLE) AS INT) AS accommodates,
        TRY_CAST(TRY_CAST(bathrooms AS DOUBLE) AS INT) AS bathrooms,
        TRY_CAST(TRY_CAST(bedrooms AS DOUBLE) AS INT) AS bedrooms,
        TRY_CAST(TRY_CAST(beds AS DOUBLE) AS INT) AS beds,
        amenities,
        TRY_CAST(REGEXP_REPLACE(price, '[\\\\$,]', '') AS DOUBLE) AS price,
        TRY_CAST(require_guest_profile_picture AS BOOLEAN) AS require_guest_profile_picture,
        TRY_CAST(require_guest_phone_verification AS BOOLEAN) AS require_guest_phone_verification, 
        TRY_CAST(month AS INT) AS month
    FROM selected_df
""")

selected_df.createOrReplaceTempView("selected_df")

null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()

+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|       0|        0|                0|                  0|            0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+



## EDA + Data Cleaning

### Dealing with outliers

In [13]:
def get_max_fence(column):
    qt = selected_df.approxQuantile(column, [0.25,0.75], 0.01)
    q1 = qt[0]
    upper = qt[1]
    iqr = upper - q1
    max_fence = upper + 1.5 * iqr
    return max_fence

#### `host_listing_count`

In [14]:
listing_count_max_fence = get_max_fence('host_listings_count')
print(listing_count_max_fence)

3.5


In [15]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE host_listings_count <= {listing_count_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

148367 rows were deleted.


In [16]:
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        CASE WHEN host_listings_count = 0.0 THEN 1.0 ELSE host_listings_count END AS host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month
    FROM selected_df
""")

#### `price`

In [17]:
price_max_fence = get_max_fence('price')
print(price_max_fence)

1274.5


In [18]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE price <= {price_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

63917 rows were deleted.


#### `property_type`

In [19]:
categories_to_append = ('Aparthotel', 'Earth house', 'Chalet', 'Cottage', 'Tiny house',
                        'Boutique hotel', 'Hotel', 'Casa particular (Cuba)', 'Bungalow',
                        'Nature lodge', 'Cabin', 'Castle', 'Treehouse', 'Island', 'Boat', 'Tent',
                        'Resort', 'Hut', 'Campsite', 'Barn', 'Dorm', 'Camper/RV', 'Farm stay', 'Yurt',
                        'Tipi', 'Pension (South Korea)', 'Dome house', 'Igloo', 'Casa particular',
                        'Houseboat', 'Lighthouse', 'Plane', 'Train', 'Parking Space')

category_list_sql = ", ".join([f"'" + c.replace("'", "''") + "'" for c in categories_to_append])
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        CASE WHEN property_type IN ({category_list_sql}) THEN 'Other' ELSE property_type END AS property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month
    FROM selected_df
""")

#### `beds`

In [20]:
beds_max_fence = get_max_fence('beds')
print(beds_max_fence)

6.0


In [21]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE beds <= {beds_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

11698 rows were deleted.


#### `amenities`

In [22]:
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        SIZE(SPLIT(amenities, ',')) + 1 AS n_amenities
    FROM selected_df
""")
selected_df.createOrReplaceTempView('selected_df')
mode_value = spark.sql("""
    SELECT n_amenities
    FROM selected_df
    WHERE amenities <> '{}'
    GROUP BY n_amenities
    ORDER BY COUNT(*) DESC
    LIMIT 1
""").first()['n_amenities']
selected_df = spark.sql(f"""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        CASE WHEN amenities = '{{}}' THEN {mode_value} ELSE n_amenities END AS n_amenities
    FROM selected_df
""")

In [23]:
amenities_max_fence = get_max_fence('n_amenities')
print(amenities_max_fence)

38.0


In [24]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE n_amenities <= {amenities_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

26744 rows were deleted.


## Encoding

In [25]:
selected_df.createOrReplaceTempView('selected_df')
df_label_encoder = spark.sql("""
    WITH property_type_index AS (
        SELECT
            property_type,
            ROW_NUMBER() OVER (ORDER BY cnt DESC, property_type ASC) - 1 AS property_type_encoded
        FROM (
            SELECT property_type, COUNT(*) AS cnt
            FROM selected_df
            GROUP BY property_type
        )
    )
    SELECT
        s.latitude,
        s.longitude,
        CASE
            WHEN s.host_is_superhost IS TRUE THEN 1
            WHEN s.host_is_superhost IS FALSE THEN 0
            ELSE CAST(s.host_is_superhost AS INT)
        END AS host_is_superhost,
        s.host_listings_count,
        p.property_type_encoded AS property_type,
        s.accommodates,
        s.bathrooms,
        s.bedrooms,
        s.beds,
        s.price,
        CASE
            WHEN s.require_guest_profile_picture IS TRUE THEN 1
            WHEN s.require_guest_profile_picture IS FALSE THEN 0
            ELSE CAST(s.require_guest_profile_picture AS INT)
        END AS require_guest_profile_picture,
        CASE
            WHEN s.require_guest_phone_verification IS TRUE THEN 1
            WHEN s.require_guest_phone_verification IS FALSE THEN 0
            ELSE CAST(s.require_guest_phone_verification AS INT)
        END AS require_guest_phone_verification,
        s.month,
        s.n_amenities
    FROM selected_df s
    LEFT JOIN property_type_index p
        ON s.property_type = p.property_type
""")
print('Columns encoded')

Columns encoded


In [ ]:
df_label_encoder.createOrReplaceTempView("df_label_encoder")
df_label_encoder = spark.sql("""
    SELECT *,
        CAST(
            CASE
                WHEN price < 100  THEN 0
                WHEN price < 200  THEN 1
                WHEN price < 400  THEN 2
                WHEN price < 700  THEN 3
                ELSE 4
            END
        AS DOUBLE) AS price_band
    FROM df_label_encoder
""")
df_label_encoder.createOrReplaceTempView("df_label_encoder")
print("Band distribution:")
spark.sql("""
    SELECT price_band, COUNT(*) AS cnt,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM df_label_encoder
    GROUP BY price_band
    ORDER BY price_band
""").show()

## Model

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor
models = [
    (
        "Linear Regression",
        LinearRegression(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            maxIter=200,
            regParam=0.01,
            elasticNetParam=0.0
        )
    ),
    (
        "Decision Tree Regressor",
        DecisionTreeRegressor(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            maxDepth=30,
            minInstancesPerNode=10,
            minInfoGain=0.0,
            maxBins=64,
            seed=SEED
        )
    ),
]

In [27]:
feature_cols = [
    "host_is_superhost",
    "host_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "month",
    "property_type",
    "require_guest_profile_picture",
    "require_guest_phone_verification",
    "n_amenities"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
prepared_df = assembler.transform(df_label_encoder)

train_prepared, test_prepared = prepared_df.randomSplit([0.8, 0.2], seed=SEED)

print(f"Linhas no Treino: {train_prepared.count()}")
print(f"Linhas no Teste: {test_prepared.count()}")

Linhas no Treino: 423648


Linhas no Teste: 105662


In [28]:
def evaluate_model(model_name: str, predictions) -> None:
    predictions = predictions.cache()
    predictions.createOrReplaceTempView("predictions")

    rmse = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="rmse",
    ).evaluate(predictions)
    mae = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="mae",
    ).evaluate(predictions)
    r2 = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="r2",
    ).evaluate(predictions)

    print(f"\n{model_name}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R2: {r2:.6f}")

    predictions.unpersist()

In [29]:
for model_name, estimator in models:
    model = estimator.fit(train_prepared)
    predictions = model.transform(test_prepared)
    evaluate_model(model_name, predictions)


Linear Regression
RMSE: 244.13
MAE: 183.72
R2: 0.286931


26/05/10 18:05:19 ERROR Executor: Exception in task 15.0 in stage 168.0 (TID 1292)
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.ml.tree.impl.DTStatsAggregator.<init>(DTStatsAggregator.scala:77)
	at org.apache.spark.ml.tree.impl.RandomForest$.$anonfun$findBestSplits$22(RandomForest.scala:681)
	at org.apache.spark.ml.tree.impl.RandomForest$.$anonfun$findBestSplits$22$adapted(RandomForest.scala:677)
	at org.apache.spark.ml.tree.impl.RandomForest$$$Lambda$6594/0x0000769565706d98.apply(Unknown Source)
	at scala.Array$.tabulate(Array.scala:443)
	at org.apache.spark.ml.tree.impl.RandomForest$.$anonfun$findBestSplits$21(RandomForest.scala:677)
	at org.apache.spark.ml.tree.impl.RandomForest$$$Lambda$6585/0x0000769565704b68.apply(Unknown Source)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2(RDD.scala:866)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2$adapted(RDD.scala:866)
	at org.apache.spark.rdd.RDD$$Lambda$2167/0x0000769564c150d0.apply(Unknown Source)
	at o

ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused

## MLP Classifier (Price Bands)

In [ ]:
mlp_assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
mlp_df = mlp_assembler.transform(df_label_encoder)
train_mlp, test_mlp = mlp_df.randomSplit([0.8, 0.2], seed=SEED)

# layers: 13 inputs → 64 → 32 → 5 price bands
mlp = MultilayerPerceptronClassifier(
    labelCol="price_band",
    featuresCol="features",
    layers=[13, 64, 32, 5],
    maxIter=200,
    blockSize=128,
    solver="l-bfgs",
    tol=1e-6,
    seed=SEED
)

mlp_model = mlp.fit(train_mlp)
mlp_predictions = mlp_model.transform(test_mlp).cache()
mlp_predictions.createOrReplaceTempView("mlp_predictions")

accuracy = MulticlassClassificationEvaluator(
    labelCol="price_band", predictionCol="prediction", metricName="accuracy"
).evaluate(mlp_predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="price_band", predictionCol="prediction", metricName="weightedF1Measure"
).evaluate(mlp_predictions)

print("\nMLP Classifier (Price Bands)")
print(f"Accuracy: {accuracy:.4f}")
print(f"Weighted F1: {f1:.4f}")

print("\nPer-band accuracy:")
spark.sql("""
    SELECT price_band,
           SUM(CASE WHEN price_band = prediction THEN 1 ELSE 0 END) AS correct,
           COUNT(*) AS total,
           ROUND(SUM(CASE WHEN price_band = prediction THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS acc_pct
    FROM mlp_predictions
    GROUP BY price_band
    ORDER BY price_band
""").show()

mlp_predictions.unpersist()